# BaddieVision - Single-Video Feature Extraction (Local)

This notebook extracts reusable artifacts for exactly one local input video:

- `tracks.csv` for shuttle detections
- `players.csv` for the current heuristic player-activity schema
- `person_tracks.jsonl` as immutable YOLO/ByteTrack observations
- `pose_cache.jsonl` as incremental MediaPipe results
- `player_assignments.jsonl` as court-aware near/far interpretation
- `metadata.json` with video and runtime details
- `preview.mp4` with shuttle and player overlays
- `<source_id>.zip` containing the full output folder

It does not run rally segmentation or shot-classifier clip extraction. The goal is one local extraction pass that produces portable per-video artifacts.


## Runtime Notes

Run this from a local Jupyter environment inside the repository checkout.

Install dependencies first if the environment is not already prepared:

```bash
python -m pip install -r requirements-dev.txt
```

The notebook expects TrackNet, pose, and in-play model assets to exist either in the checkout already or under an extracted local model bundle path. InpaintNet assets are required only for explicit legacy opt-in.
- FFmpeg is optional: preparation tries CUDA/NVENC, CPU FFmpeg, then optimized OpenCV automatically.


In [ ]:
from pathlib import Path

REPO_DIR = Path("/home/max/code/BaddieVision")

# Required.
INPUT_VIDEO_PATH = "/home/max/code/BaddieInput/malaysia_max.mp4"  # Example: /home/max/code/videos/my_match.mp4

# Optional overrides.
SOURCE_ID = ""
USE_INPAINTNET = False  # Legacy comparison only; maintained extraction is TrackNet-only.
START_TIME_SEC = 30  # Example: 120 for 2:00 into the video.
END_TIME_SEC = 120  # Example: 360 for 6:00 into the video. Leave as None to use the rest of the file.
WORKING_FPS = 30  # Downsample the working clip to this FPS. Set to None to keep the original FPS.
PERSON_YOLO_MODEL = "yolov8n.pt"
COURT_CALIBRATION = "/home/max/code/BaddieInput/malaysiamax.json"  # Optional path to one calibration JSON for player extraction heuristics.
MODELS_BUNDLE_ROOT = "/home/max/code/BaddieInput/badminton-models-v1-2026-07-01"  # Optional extracted bundle root containing models/, InPlay/models/, and src/TrackNetV3/ckpts/.
TRACKNET_MAX_SAMPLE_NUM = 300
TRACKNET_VIDEO_RANGE = None  # Example: (0, 45)
OUTPUT_ROOT = REPO_DIR / "outputs"

print(f"Repository root: {REPO_DIR}")
print(f"Output root: {OUTPUT_ROOT}")


Repository root: /home/max/code/BaddieVision
Output root: /home/max/code/BaddieVision/outputs


## Load Imports

This also puts the repo and TrackNet submodule on `sys.path`.


In [2]:
import shutil
import sys
import zipfile

if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

src_dir = REPO_DIR / "src"
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

tracknet_dir = src_dir / "TrackNetV3"
if str(tracknet_dir) not in sys.path:
    sys.path.insert(0, str(tracknet_dir))

tracknet_predict_args = tracknet_dir / "predictArgs.py"
predict_args_source = tracknet_predict_args.read_text(encoding="utf-8")
if "def _extend_prediction_dict(target, source):" not in predict_args_source:
    raise RuntimeError(
        "Stale TrackNetV3 checkout: src/TrackNetV3/predictArgs.py is missing the PeakValue fix. "
        "Update the TrackNetV3 submodule before running this notebook."
    )

from InPlay.heuristic.config import HeuristicConfig
from InPlay.heuristic.person_tracks import extract_person_tracks
from InPlay.heuristic.player_interpretation import interpret_players
from InPlay.heuristic.tracks import read_track_csv
from TrackNetV3.predictArgs import run_prediction
from src.pose_estimator import ensure_pose_model_asset, pose_backend_info, pose_connections
# Reload repo-local helpers so this cell also works after updating files in a live kernel.
import importlib
import src.single_video.artifacts as single_video_artifacts
import src.single_video as single_video_helpers
importlib.reload(single_video_artifacts)
importlib.reload(single_video_helpers)
from src.single_video import (
    copy_model_tree,
    load_track_models,
    looks_like_model_root,
    prepare_working_video,
    read_video_info,
    render_player_preview,
    link_shuttle_candidates,
    link_shuttle_hypotheses,
    segment_suffix,
    write_metadata,
    write_result_manifest,
    zip_result_dir,
)

POSE_MODEL_ASSET = ensure_pose_model_asset()
POSE_BACKEND = pose_backend_info(POSE_MODEL_ASSET)
print(f"Pose model asset ready: {POSE_MODEL_ASSET}")


Pose model asset ready: /home/max/code/BaddieVision/models/pose_landmarker_full.task


## Local adapters

Resolve local paths and restore model assets while shared processing helpers come from `src.single_video`.


In [3]:
def ensure_local_models() -> None:
    if looks_like_model_root(REPO_DIR, use_inpaintnet=USE_INPAINTNET):
        print("Model files already present in the checkout.")
        return
    if not MODELS_BUNDLE_ROOT:
        raise FileNotFoundError(
            "Required model files are missing from the checkout. Set MODELS_BUNDLE_ROOT to an extracted local bundle root."
        )
    bundle_root = Path(MODELS_BUNDLE_ROOT).expanduser().resolve()
    copy_model_tree(bundle_root, REPO_DIR, use_inpaintnet=USE_INPAINTNET)


def resolve_input_video() -> Path:
    if not INPUT_VIDEO_PATH:
        raise ValueError("Set INPUT_VIDEO_PATH before running extraction.")
    video_path = Path(INPUT_VIDEO_PATH).expanduser().resolve()
    if not video_path.is_file():
        raise FileNotFoundError(video_path)
    return video_path


## Prepare Output Folder

This creates `outputs/<source_id>/`, optionally clips the video to a requested time range, optionally downsamples it, and uses that working video for extraction.


In [4]:
ensure_local_models()
video_path = resolve_input_video()
original_video_info = read_video_info(video_path)
clip_suffix = segment_suffix(START_TIME_SEC, END_TIME_SEC)
default_source_id = video_path.stem if clip_suffix == "full" else f"{video_path.stem}_{clip_suffix}"
source_id = SOURCE_ID or default_source_id
result_dir = OUTPUT_ROOT / source_id
result_dir.mkdir(parents=True, exist_ok=True)

working_video_path = result_dir / f"{source_id}_input.mp4"
local_video_path = prepare_working_video(video_path, working_video_path, START_TIME_SEC, END_TIME_SEC, WORKING_FPS)

print(f"Original video: {video_path}")
print(f"Working video: {local_video_path}")
print(f"Original FPS: {original_video_info['fps']}")
print(f"Source ID: {source_id}")
print(f"Result dir: {result_dir}")


Model files already present in the checkout.
Prepared working video with stream-copy-trim: /home/max/code/BaddieVision/outputs/malaysia_max_30s_to_120s/malaysia_max_30s_to_120s_input.mp4
Original video: /home/max/code/BaddieInput/malaysia_max.mp4
Working video: /home/max/code/BaddieVision/outputs/malaysia_max_30s_to_120s/malaysia_max_30s_to_120s_input.mp4
Original FPS: 59.94005994005994
Source ID: malaysia_max_30s_to_120s
Result dir: /home/max/code/BaddieVision/outputs/malaysia_max_30s_to_120s


## Run Extraction

This is the main GPU work: shuttle extraction, player extraction, validation, preview rendering, metadata, and ZIP creation. It does not run rally segmentation.


In [5]:
video_info = read_video_info(local_video_path)
models = load_track_models(REPO_DIR, use_inpaintnet=USE_INPAINTNET)

tracks_path = result_dir / "tracks.csv"
candidates_path = result_dir / "shuttle_candidates.jsonl"
tracklets_path = result_dir / "shuttle_tracklets.jsonl"
hypotheses_path = result_dir / "shuttle_hypotheses.jsonl"
players_path = result_dir / "players.csv"
person_tracks_path = result_dir / "person_tracks.jsonl"
pose_cache_path = result_dir / "pose_cache.jsonl"
assignments_path = result_dir / "player_assignments.jsonl"
metadata_path = result_dir / "metadata.json"
preview_path = result_dir / "preview.mp4"
manifest_json_path = result_dir / "artifact_manifest.json"
manifest_txt_path = result_dir / "artifact_manifest.txt"
zip_path = result_dir / f"{source_id}.zip"


In [6]:

print("[1/6] Starting TrackNet shuttle tracking...")
run_prediction(
    video_file=str(local_video_path),
    save_dir=str(result_dir),
    batch_size=12,
    output_video=False,
    large_video=True,
    max_sample_num=TRACKNET_MAX_SAMPLE_NUM,
    video_range=TRACKNET_VIDEO_RANGE,
    optimized_inference=True,
    chunk_size_frames=4096,
    num_workers=4,
    pin_memory=True,
    candidate_output_path=str(candidates_path),
    reuse_models=models,
)
print("[1/6] TrackNet shuttle tracking complete.")

tracknet_csv = result_dir / f"{local_video_path.stem.lower()}_ball.csv"
if not tracknet_csv.is_file():
    raise FileNotFoundError(f"TrackNet output missing: {tracknet_csv}")
shutil.move(tracknet_csv, tracks_path)
link_shuttle_candidates(candidates_path, tracklets_path)
print(f"[2/6] Shuttle tracks saved to {tracks_path}")


[1/6] Starting TrackNet shuttle tracking...
Generate median image...


Sampling median frames:  99%|█████████▉| 300/302 [00:05<00:00, 51.24frame/s]


Median image generated.
Video length: 5423


100%|██████████| 448/448 [00:04<00:00, 89.88it/s] 


ValueError: candidate frames do not completely cover expected inclusive working-video range [0, 5422]; missing=[5387, 5388, 5389, 5390, 5391, 5392, 5393, 5394, 5395, 5396] (+26 more), unexpected=[]

In [ ]:

print("[3/6] Extracting immutable YOLO/ByteTrack person observations...")
extract_person_tracks(local_video_path, person_tracks_path, PERSON_YOLO_MODEL)
interpret_players(
    local_video_path, person_tracks_path, COURT_CALIBRATION, pose_cache_path,
    assignments_path, players_path, pose_model_asset=POSE_MODEL_ASSET,
)
print("[3/6] Court-aware player interpretation and pose enrichment complete.")
link_shuttle_hypotheses(candidates_path, tracklets_path, hypotheses_path)
print(f"[3/6] Shuttle association hypotheses saved to {hypotheses_path}")

print("[4/6] Validating shuttle tracks and rendering preview...")
read_track_csv(tracks_path, (video_info["width"], video_info["height"]), HeuristicConfig())
render_player_preview(
    local_video_path, tracks_path, person_tracks_path, assignments_path,
    pose_cache_path, preview_path, pose_connections(), candidate_path=candidates_path, tracklet_path=tracklets_path, hypotheses_path=hypotheses_path,
)
print(f"[4/6] Preview complete: {preview_path}")

print("[5/6] Writing metadata and artifact manifest...")
write_metadata(
    metadata_path,
    source_id,
    video_path,
    local_video_path,
    original_video_info,
    video_info,
    models,
    START_TIME_SEC,
    END_TIME_SEC,
    WORKING_FPS,
    player_detector=PERSON_YOLO_MODEL,
    pose_backend=POSE_BACKEND,
)
artifact_paths = [
    local_video_path,
    tracks_path,
    candidates_path,
    tracklets_path,
    hypotheses_path,
    players_path,
    person_tracks_path,
    pose_cache_path,
    assignments_path,
    metadata_path,
    preview_path,
]
manifest = write_result_manifest(manifest_json_path, result_dir, source_id, artifact_paths)
manifest_lines = [f"Source ID: {source_id}", f"Result dir: {result_dir}", "Artifacts:"]
for item in manifest["files"]:
    manifest_lines.append(f"- {item['relative_path']} ({item['size_bytes']} bytes)")
manifest_txt_path.write_text("\n".join(manifest_lines) + "\n", encoding="utf-8")
print("[5/6] Metadata and manifest complete.")

print("[6/6] Creating portable ZIP bundle...")
zip_result_dir(result_dir, zip_path)
print(f"[6/6] ZIP complete: {zip_path}")

print(manifest_txt_path.read_text(encoding="utf-8"))
print(f"- {manifest_json_path}")
print(f"- {manifest_txt_path}")
print(f"- {zip_path}")


## Inspect Results


In [ ]:
print((result_dir / "artifact_manifest.txt").read_text(encoding="utf-8"))
json.loads((result_dir / "artifact_manifest.json").read_text(encoding="utf-8"))
